## 1. Loading the data

In [1]:
import pandas as pd

import sys
sys.path.insert(0, '..')  # points to fpl-engine/ from notebooks/

from fpl_engine.api_client import (load_bootstrap, load_fixtures, load_player_history)
from fpl_engine.data_dictionary import (bootstrap_static_elements_data_dictionary, player_summary_data_dictionary, fixtures_data_dictionary)
from fpl_engine.pipeline import (build_raw_modelling_feature_store)

bootstrap = load_bootstrap()
fixtures = load_fixtures()

number_of_players = len([p["id"] for p in bootstrap["elements"]])

# for i in range(1,number_of_players):
#     load_player_history(i)
#     if i % 50 == 0:
#         print(f"{i}/{number_of_players} players loaded")
# print(f"Done - all player histories saved")

Loaded 841 players
Loaded 20 teams
Loaded 380 fixtures


In [ ]:
players_df = pd.DataFrame(bootstrap['elements'])
print(players_df.shape)
print(players_df.columns.tolist())
players_df["second_name"]

(841, 105)
['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_intercep

0                Raya Martín
1      Arrizabalaga Revuelta
2                       Hein
3                    Setford
4       dos Santos Magalhães
               ...          
836                    Abbey
837                    Gomes
838                Armstrong
839                   Brooks
840                   Gracey
Name: second_name, Length: 841, dtype: str

In [20]:
import json
from pathlib import Path
RAW_DIR = Path.cwd().parent / "Data" / "raw"

salah_id = players_df[players_df["second_name"] == "Salah"]["id"].values[0]
with open(RAW_DIR / "players" / f"{salah_id}.json") as f:
    salah_data = json.load(f)

history_df = pd.DataFrame(salah_data['history'])
print(history_df.columns.tolist())
history_df[['round' ,'total_points', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'ict_index', 'selected', 'value']].head(10)
history_df["total_points"].sum()


['element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'kickoff_time', 'team_h_score', 'team_a_score', 'round', 'modified', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries', 'tackles', 'defensive_contribution', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value', 'transfers_balance', 'selected', 'transfers_in', 'transfers_out']


np.int64(123)

In [18]:
players_df[players_df["second_name"] == "Salah"][['second_name', 'id', 'element_type', "total_points"]]


,second_name,id,element_type,total_points
471,Salah,381,3,123


In [37]:
salah_id = players_df[players_df["second_name"] == "Salah"]["id"].values[0]
with open(RAW_DIR / "players" / f"{salah_id}.json") as f:
    salah_data = json.load(f)

history_df = pd.DataFrame(salah_data["history"])
print(history_df.columns.tolist())
print(salah_id)
history_df["element"]

['element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'kickoff_time', 'team_h_score', 'team_a_score', 'round', 'modified', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries', 'tackles', 'defensive_contribution', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value', 'transfers_balance', 'selected', 'transfers_in', 'transfers_out']
381


0     381
1     381
2     381
3     381
4     381
5     381
6     381
7     381
8     381
9     381
10    381
11    381
12    381
13    381
14    381
15    381
16    381
17    381
18    381
19    381
20    381
21    381
22    381
23    381
24    381
25    381
26    381
27    381
28    381
29    381
30    381
31    381
32    381
33    381
34    381
35    381
36    381
37    381
Name: element, dtype: int64

## 2. Data Cleaning

In [2]:
df = pd.read_parquet("../Data/processed/raw_modelling_feature_store.parquet")
df.head(10)

,player_id,gameweek,total_points,minutes,fixture_id,was_home,opponent_team,starts,price,selected,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,team,position,web_name,difficulty,team_name,opponent_team_name
0,1,1,10,90,9,False,14,1,5.5,1531911,...,0.0,0.00,0.00,1.52,1,1,Raya,4,Arsenal,Man Utd
1,1,2,6,90,11,True,11,1,5.5,2284634,...,0.0,0.00,0.00,0.17,1,1,Raya,2,Arsenal,Leeds
2,1,3,2,90,25,False,12,1,5.5,2406964,...,0.0,0.02,0.02,0.52,1,1,Raya,4,Arsenal,Liverpool
3,1,4,6,90,31,True,16,1,5.5,2765759,...,0.0,0.00,0.00,0.20,1,1,Raya,3,Arsenal,Nott'm Forest
4,1,5,2,90,41,True,13,1,5.5,2762632,...,0.0,0.01,0.01,0.89,1,1,Raya,4,Arsenal,Man City
5,1,6,2,90,58,False,15,1,5.6,3029654,...,0.0,0.01,0.01,0.61,1,1,Raya,3,Arsenal,Newcastle
6,1,7,6,90,61,True,19,1,5.6,3297838,...,0.0,0.00,0.00,0.49,1,1,Raya,2,Arsenal,West Ham
7,1,8,6,90,74,False,10,1,5.7,3447590,...,0.0,0.00,0.00,0.44,1,1,Raya,3,Arsenal,Fulham
8,1,9,6,90,81,True,8,1,5.7,3546345,...,0.0,0.00,0.00,0.47,1,1,Raya,3,Arsenal,Crystal Palace
9,1,10,6,90,92,False,3,1,5.8,3816996,...,0.0,0.00,0.00,0.42,1,1,Raya,2,Arsenal,Burnley
